# Protected areas code refactoring
### Week 7 - Discussion section (presenter manual)

This notebook walks through the whole discussion section in order. It shows the colleague's **original code** first, exactly as students receive it, and then the **refactored code**, the finished percentage calculation and the data-quality investigation. Presenter notes in grey blockquotes are talking points for class.

**Data.** A subset of the November 2025 World Database on Protected Areas (WDPA) with only polygons for protected areas in Latin America, from the course Google Drive (`WDPA_Nov2025_Latam.zip`). It is unzipped at `~/Desktop/eds220-discussion-sections/data/WDPA_Nov2025_Latam/`, and the setup cell points `data_folder` there.

**References**
- UNEP-WCMC and IUCN (2025), *Protected Planet: The World Database on Protected Areas (WDPA)*, November 2025. Cambridge, UK. www.protectedplanet.net
- UNEP-WCMC (2019), *User Manual for the World Database on Protected Areas and world database on other effective area-based conservation measures: 1.6*. Cambridge, UK. http://wcmc.io/WDPA_Manual (`WDPA_WDOECM_Manual_1_6.pdf` in the data folder)

## Setup: the colleague's data import

This cell is the student notebook's code, with `data_folder` pointed at the data on the Desktop.

In [1]:
import os
import pandas as pd
import geopandas as gpd

pd.set_option('display.max_columns', None)

# ------ IMPORT DATASETS ------

data_folder = os.path.join(os.path.expanduser('~'), 'Desktop', 'eds220-discussion-sections',
                           'data', 'WDPA_Nov2025_Latam')

fp = os.path.join(data_folder, 'WDPA_Nov2025_Latam_shp_0', 'WDPA_Nov2025_Latam_shp0-polygons.shp')
wdpa0 = gpd.read_file(fp)

fp = os.path.join(data_folder, 'WDPA_Nov2025_Latam_shp_1', 'WDPA_Nov2025_Latam_shp1-polygons.shp')
wdpa1 = gpd.read_file(fp)

fp = os.path.join(data_folder, 'WDPA_Nov2025_Latam_shp_2', 'WDPA_Nov2025_Latam_shp2-polygons.shp')
wdpa2 = gpd.read_file(fp)

# Standardize column names to lowercase
wdpa0.columns = wdpa0.columns.str.lower()
wdpa1.columns = wdpa1.columns.str.lower()
wdpa2.columns = wdpa2.columns.str.lower()

## 1. Metadata exploration

The data folder has the user manual (`WDPA_WDOECM_Manual_1_6.pdf`), a one-page attribute summary (`Summary_table_WDPA_WDOECM_attributes.pdf`), and `Shapefile_splitting_README.txt`, which explains that the download is split into three shapefiles that should be merged into one layer.

The columns this analysis depends on:

| Column | Meaning |
|---|---|
| `wdpaid`, `wdpa_pid` | Site identifier, and parcel identifier for sites split into several polygons |
| `name`, `desig_eng` | Name and designation in English (National Park, Ramsar Site, ...) |
| `desig_type` | National, Regional or International designation |
| `marine` | `'0'` = predominantly terrestrial (10% or less marine), `'1'` = part marine and part terrestrial, `'2'` = predominantly marine (90% or more). Stored as **text** |
| `rep_area`, `rep_m_area` | Total and marine area in km² **reported by the data provider** |
| `gis_area`, `gis_m_area` | Total and marine area in km² **calculated by UNEP-WCMC from the polygon** in the Mollweide projection |
| `status` | Designated, Inscribed, Established, Adopted, Proposed, ... |
| `iso3` | Country code; transboundary sites carry several codes separated by `;` |

In [2]:
for i, df in enumerate([wdpa0, wdpa1, wdpa2]):
    print(f"wdpa{i}: {df.shape[0]:,} rows, {df.shape[1]} columns, CRS {df.crs}")

wdpa0.head(3)

wdpa0: 3,679 rows, 31 columns, CRS EPSG:4326
wdpa1: 3,421 rows, 31 columns, CRS EPSG:4326
wdpa2: 3,276 rows, 31 columns, CRS EPSG:4326


,wdpaid,wdpa_pid,pa_def,name,orig_name,desig,desig_eng,desig_type,iucn_cat,int_crit,marine,rep_m_area,gis_m_area,rep_area,gis_area,no_take,no_tk_area,status,status_yr,gov_type,own_type,mang_auth,mang_plan,verif,metadataid,sub_loc,parent_iso,iso3,supp_info,cons_obj,geometry
0,1.0,1,1,Diamond Reef and Salt Fish Tail Reef,Diamond Reef,Marine Reserve,Marine Reserve,National,Ia,Not Applicable,2,14.572487,14.587808,14.581467,14.587808,All,14.572487,Designated,1973,Federal or national ministry or agency,State,Fisheries Division,Not Reported,State Verified,1807,AG-04,ATG,ATG,Not Applicable,Not Applicable,"POLYGON ((-61.82494 17.18497, -61.82497 17.184..."
1,2.0,2,1,Palaster Reef,Palaster Reef,Marine Reserve,Marine Reserve,National,Ia,Not Applicable,2,3.829107,3.833094,3.831674,3.833094,All,3.829107,Designated,1973,Federal or national ministry or agency,State,Fisheries Division,Not Reported,State Verified,1807,AG-10,ATG,ATG,Not Applicable,Not Applicable,"POLYGON ((-61.74007 17.52001, -61.77174 17.526..."
2,3.0,3,1,Laguna de los Pozuelos,Laguna de los Pozuelos,Monumento Natural,Nature Monument,National,III,Not Applicable,0,0.000000,0.000000,160.000000,217.931016,Not Applicable,0.000000,Designated,1980,Federal or national ministry or agency,Not Reported,Administración de Parques Nacionales,Not Reported,State Verified,1935,AR-Y,ARG,ARG,Not Applicable,Not Applicable,"POLYGON ((-65.98955 -22.47423, -65.99441 -22.4..."


In [3]:
print(wdpa0['marine'].value_counts(), '\n')
print(wdpa0['status'].value_counts())

marine
0    3213
1     289
2     177
Name: count, dtype: int64 

status
Designated     3374
Proposed        275
Inscribed        29
Established       1
Name: count, dtype: int64


> **Presenter note.** Then open the data viewer at www.protectedplanet.net and zoom into Central America. Ask students to look at Costa Rica: the large polygon around Isla del Coco in the Pacific, far from the mainland, is a good preview of section 5.

## 2. Plan refactoring

### The original code


In [2]:
# Calculate total protected reported area for each Central American country from WDPA datasets

# Standardize column names to lowercase
wdpa0.columns = wdpa0.columns.str.lower()
wdpa1.columns = wdpa1.columns.str.lower()
wdpa2.columns = wdpa2.columns.str.lower()

# Belize
area0 = wdpa0.loc[wdpa0["iso3"] == "BLZ", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "BLZ", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "BLZ", "rep_area"].sum()
total_area_blz = area0 + area1 + area2
print(f"Total protected area in Belize (sq. km): {total_area_blz:,.2f}")

# Costa Rica
area0 = wdpa0.loc[wdpa0["iso3"] == "CRI", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "CRI", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "CRI", "rep_area"].sum()
total_area_cri = area0 + area1 + area2
print(f"Total protected area in Costa Rica (sq. km): {total_area_cri:,.2f}")

# El Salvador
area0 = wdpa0.loc[wdpa0["iso3"] == "SLV", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "SLV", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "SLV", "rep_area"].sum()
total_area_slv = area0 + area1 + area2
print(f"Total protected area in El Salvador (sq. km): {total_area_slv:,.2f}")

# Guatemala
area0 = wdpa0.loc[wdpa0["iso3"] == "GTM", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "GTM", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "GTM", "rep_area"].sum()
total_area_gtm = area0 + area1 + area2
print(f"Total protected area in Guatemala (sq. km): {total_area_gtm:,.2f}")

# Honduras
area0 = wdpa0.loc[wdpa0["iso3"] == "HND", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "HND", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "HND", "rep_area"].sum()
total_area_hnd = area0 + area1 + area2
print(f"Total protected area in Honduras (sq. km): {total_area_hnd:,.2f}")

# Nicaragua
area0 = wdpa0.loc[wdpa0["iso3"] == "NIC", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "NIC", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "NIC", "rep_area"].sum()
total_area_nic = area0 + area1 + area2
print(f"Total protected area in Nicaragua (sq. km): {total_area_nic:,.2f}")

# Panama
area0 = wdpa0.loc[wdpa0["iso3"] == "PAN", "rep_area"].sum()
area1 = wdpa1.loc[wdpa1["iso3"] == "PAN", "rep_area"].sum()
area2 = wdpa2.loc[wdpa2["iso3"] == "PAN", "rep_area"].sum()
total_area_pan = area0 + area1 + area2
print(f"Total protected area in Panama (sq. km): {total_area_pan:,.2f}")

Total protected area in Belize (sq. km): 10,181.87
Total protected area in Costa Rica (sq. km): 199,170.72
Total protected area in El Salvador (sq. km): 5,701.12
Total protected area in Guatemala (sq. km): 40,316.17
Total protected area in Honduras (sq. km): 96,103.89
Total protected area in Nicaragua (sq. km): 84,985.39
Total protected area in Panama (sq. km): 133,553.40


In [3]:
# ----------------------------------------------------------------
# Calculate percentage of protected   area for Belize
pct_blz = (total_area_blz / 22966) * 100
print(f"Percentage of protected area in Belize relative to country's land area: {pct_blz:.2f}%\n")

Percentage of protected area in Belize relative to country's land area: 44.33%



### What makes this code hard to update

1. **The file paths are hard-coded three times**, differing only in the index `0`, `1`, `2`.
2. **The column renaming is done twice** (once at import, again at the top of the analysis), and three times each.
3. **One block is copied seven times.** Each country repeats the same three-line filter and sum, with its ISO3 code and name typed by hand, so a fix has to be made in seven places, and a typo in one copy (say `"CRI"` in the Panama block) would go unnoticed.
4. **The data lives in the wrong places.** Land areas sit only in a markdown cell, and the one percentage hard-codes `22966`.
5. **Fourteen loose variables** (`area0`, ..., `total_area_pan`) hold results that belong in one table, and `area0`/`area1`/`area2` are overwritten in every block.

### High-level refactoring plan

1. Load the three shapefiles **in a loop**, lowercase the columns once, and **concatenate** them into one GeoDataFrame, which is what the README asks for. Check that no rows are lost.
2. Store the countries in **one table** indexed by ISO3, with name and land area.
3. Compute protected area for all countries at once with **`groupby('iso3')`** and add it as a column.

> **Presenter note.** Have groups write their plan before showing this one. A common alternative keeps the three files separate and writes a function that sums `rep_area` over a list of GeoDataFrames for one country code. Both are fine; concatenating is simpler because every later step then deals with one table.

## 3. Refactor!

### Load and combine the data

In [9]:
def load_wdpa(data_folder, n_parts=3):
    """Read the split WDPA polygon shapefiles and merge them into one GeoDataFrame.

    The download is split into `n_parts` shapefiles, one per folder
    WDPA_Nov2025_Latam_shp_{i}/WDPA_Nov2025_Latam_shp{i}-polygons.shp.
    Column names are lowercased.
    """
    parts = []
    for i in range(n_parts):
        fp = os.path.join(data_folder,
                          f'WDPA_Nov2025_Latam_shp_{i}',
                          f'WDPA_Nov2025_Latam_shp{i}-polygons.shp')
        gdf = gpd.read_file(fp)
        gdf.columns = gdf.columns.str.lower()
        parts.append(gdf)

    wdpa = pd.concat(parts, ignore_index=True)

    # No rows lost or duplicated by the concatenation
    assert len(wdpa) == sum(len(gdf) for gdf in parts)
    return wdpa


wdpa = load_wdpa(data_folder)
print(f"{len(wdpa):,} protected-area polygons, CRS {wdpa.crs}")

10,376 protected-area polygons, CRS EPSG:4326


### One table for the countries

Land areas in km² from Wikipedia (November 5, 2025), as given by the colleague.

In [10]:
cam = pd.DataFrame({
    'iso3': ['BLZ', 'CRI', 'SLV', 'GTM', 'HND', 'NIC', 'PAN'],
    'country': ['Belize', 'Costa Rica', 'El Salvador', 'Guatemala',
                'Honduras', 'Nicaragua', 'Panama'],
    'land_area_sqkm': [22966, 51180, 21041, 108889, 112492, 130375, 75417],
}).set_index('iso3')

cam

,country,land_area_sqkm
iso3,,
BLZ,Belize,22966
CRI,Costa Rica,51180
SLV,El Salvador,21041
GTM,Guatemala,108889
HND,Honduras,112492
NIC,Nicaragua,130375
PAN,Panama,75417


### Total protected area, all countries at once

In [11]:
# Sum the reported area by country code, for every country in the data
protected_by_country = wdpa.groupby('iso3')['rep_area'].sum()

# Keep the Central American countries (a country with no polygons would get 0)
cam['protected_area_sqkm'] = protected_by_country.reindex(cam.index, fill_value=0)

for row in cam.itertuples():
    print(f"Total protected area in {row.country} (sq. km): {row.protected_area_sqkm:,.2f}")

Total protected area in Belize (sq. km): 10,181.87
Total protected area in Costa Rica (sq. km): 199,170.72
Total protected area in El Salvador (sq. km): 5,701.12
Total protected area in Guatemala (sq. km): 40,316.17
Total protected area in Honduras (sq. km): 96,103.89
Total protected area in Nicaragua (sq. km): 84,985.39
Total protected area in Panama (sq. km): 133,553.40


Check that the refactor did not change what the code does: the totals must match the original output.

In [9]:
original_totals = pd.Series({'BLZ': total_area_blz, 'CRI': total_area_cri, 'SLV': total_area_slv,
                             'GTM': total_area_gtm, 'HND': total_area_hnd, 'NIC': total_area_nic,
                             'PAN': total_area_pan})

assert ((cam['protected_area_sqkm'] - original_totals).abs() < 1e-6).all()
print("Refactored totals match the original code.")

Refactored totals match the original code.


> **Presenter note.** This check is part of the lesson: refactoring means the output does not change, and a one-line assertion proves it rather than eyeballing seven printed numbers.

## 4. Finish percentage coverage

With land areas in the same table, the colleague's unfinished TO DO becomes one line instead of seven more copies.

In [10]:
cam['pct_protected'] = cam['protected_area_sqkm'] / cam['land_area_sqkm'] * 100

for row in cam.itertuples():
    print(f"Total protected area in {row.country} (sq. km): {row.protected_area_sqkm:,.2f}")
    print(f"Percentage of protected area in {row.country} relative to country's land area: "
          f"{row.pct_protected:.2f}%\n")

Total protected area in Belize (sq. km): 10,181.87
Percentage of protected area in Belize relative to country's land area: 44.33%

Total protected area in Costa Rica (sq. km): 199,170.72
Percentage of protected area in Costa Rica relative to country's land area: 389.16%

Total protected area in El Salvador (sq. km): 5,701.12
Percentage of protected area in El Salvador relative to country's land area: 27.10%

Total protected area in Guatemala (sq. km): 40,316.17
Percentage of protected area in Guatemala relative to country's land area: 37.03%

Total protected area in Honduras (sq. km): 96,103.89
Percentage of protected area in Honduras relative to country's land area: 85.43%

Total protected area in Nicaragua (sq. km): 84,985.39
Percentage of protected area in Nicaragua relative to country's land area: 65.19%

Total protected area in Panama (sq. km): 133,553.40
Percentage of protected area in Panama relative to country's land area: 177.09%



In [11]:
# Belize matches the colleague's one hand-written percentage
assert abs(cam.at['BLZ', 'pct_protected'] - pct_blz) < 1e-9

## 5. Investigate issues

### Do these percentages make sense?

No. **Costa Rica and Panama come out above 100%**, which is impossible for a share of land area, and the other countries are inflated too. The cells below show why, one problem at a time.

In [12]:
cam_pa = wdpa[wdpa['iso3'].isin(cam.index)].copy()
print(f"{len(cam_pa):,} polygons in the seven Central American countries")

1,164 polygons in the seven Central American countries


### Problem 1: marine area is divided by land area

`rep_area` is the **total** area, sea included, while the denominator is land only.

In [13]:
# Reported area by country and marine class ('0' terrestrial, '1' coastal, '2' marine)
pd.crosstab(cam_pa['iso3'], cam_pa['marine'],
            values=cam_pa['rep_area'], aggfunc='sum').fillna(0).round(0)

marine,0,1,2
iso3,,,
BLZ,8398.0,0.0,1784.0
CRI,24866.0,7288.0,167017.0
GTM,37166.0,3150.0,0.0
HND,74616.0,3385.0,18102.0
NIC,61905.0,21537.0,1543.0
PAN,39070.0,11504.0,82980.0
SLV,2625.0,2649.0,426.0


In [14]:
# The largest sites in Costa Rica
(cam_pa[cam_pa['iso3'] == 'CRI']
 .sort_values('rep_area', ascending=False)
 [['name', 'desig_eng', 'marine', 'status', 'rep_area', 'gis_m_area']]
 .head(6))

,name,desig_eng,marine,status,rep_area,gis_m_area
8738,Montes Submarinos,Marine Management Area,2,Designated,106331.964000,106283.640693
123,Isla del Coco,National Park,2,Designated,54853.528610,54844.087019
324,La Amistad,UNESCO-MAB Biosphere Reserve,0,Designated,5845.920000,0.000000
8686,Savegre,UNESCO-MAB Biosphere Reserve,1,Designated,3129.140000,1493.303920
2393,Cocos Island National Park,World Heritage Site (natural or mixed),2,Inscribed,1997.000000,1863.902651
259,Internacional La Amistad,National Park,0,Designated,1982.899276,0.000000


> **Presenter note.** Two marine sites, Montes Submarinos (seamounts) and Isla del Coco, are each larger than Costa Rica's whole land area. This is what drives Costa Rica past 100%.

### Problem 2: overlapping sites are counted more than once

The same ground often carries several designations, for example a National Park that is also a World Heritage Site or a Ramsar Site. The manual's guidance is to build a "flat" layer with no overlaps before computing coverage, and it warns that summing an area field is almost never a suitable way to measure a group of sites.

In [15]:
# Isla del Coco appears three times: National Park, World Heritage Site and Ramsar Site
cam_pa.loc[cam_pa['name'].str.contains('Isla del Coco|Cocos Island'),
           ['name', 'desig_eng', 'desig_type', 'marine', 'rep_area']]

,name,desig_eng,desig_type,marine,rep_area
123,Isla del Coco,National Park,National,2,54853.52861
2393,Cocos Island National Park,World Heritage Site (natural or mixed),International,2,1997.00000
2448,Isla del Coco,"Ramsar Site, Wetland of International Importance",International,2,996.23000


In [16]:
# How much do overlaps inflate the total? Compare summing polygon areas with the area
# of the dissolved ("flat") layer, both measured in an equal-area projection
EQUAL_AREA = 'ESRI:54009'   # World Mollweide, the projection WDPA uses for gis_area

summed = cam_pa.to_crs(EQUAL_AREA).area.groupby(cam_pa['iso3']).sum() / 1e6
flat = cam_pa.dissolve(by='iso3').to_crs(EQUAL_AREA).area / 1e6

overlap = pd.DataFrame({'summed_sqkm': summed, 'dissolved_sqkm': flat})
overlap['double_counted_sqkm'] = overlap['summed_sqkm'] - overlap['dissolved_sqkm']

assert (overlap['dissolved_sqkm'] <= overlap['summed_sqkm'] + 1e-6).all()
overlap.round(0)

,summed_sqkm,dissolved_sqkm,double_counted_sqkm
iso3,,,
BLZ,17275.0,15034.0,2241.0
CRI,216984.0,198422.0,18562.0
GTM,41829.0,35363.0,6466.0
HND,64920.0,51082.0,13838.0
NIC,85355.0,57003.0,28352.0
PAN,129319.0,111233.0,18086.0
SLV,6361.0,4757.0,1604.0


### Problem 3: not every record is an established protected area

The manual's section on *Status* notes that some records are only **Proposed**. For legally designated sites only, it recommends removing `Proposed`, `Established` and `Not Reported` records. Users interested in any site meeting the protected-area definition should keep `Established` sites.

In [17]:
print(cam_pa['status'].value_counts(), '\n')
print("Reported area of Proposed sites (sq. km):",
      f"{cam_pa.loc[cam_pa['status'] == 'Proposed', 'rep_area'].sum():,.0f}")

status
Designated    1071
Proposed        86
Inscribed        7
Name: count, dtype: int64 

Reported area of Proposed sites (sq. km): 16,472


### Problem 4: reported area and polygon area disagree

`rep_area` comes from the data provider and `gis_area` from the polygon. The manual recommends `gis_area` (or measuring the polygon yourself) for any analysis about the spatial data.

In [18]:
cam_pa.groupby('iso3')[['rep_area', 'gis_area']].sum().round(0)

,rep_area,gis_area
iso3,,
BLZ,10182.0,17179.0
CRI,199171.0,214885.0
GTM,40316.0,41590.0
HND,96104.0,64552.0
NIC,84985.0,84841.0
PAN,133553.0,128480.0
SLV,5701.0,6323.0


In [19]:
# One reason: sites split into several parcels can report 0 area on every parcel
parcels = cam_pa[cam_pa['wdpaid'].duplicated(keep=False)]
print(f"{parcels['wdpaid'].nunique()} sites are split into {len(parcels)} parcels")
parcels.groupby(['iso3', 'name'])[['rep_area', 'gis_area']].sum().round(1).head(10)

10 sites are split into 62 parcels


rep_area  gis_area
iso3 name                                                   
BLZ  Bacalar Chico                             0.0      60.0
     Caye Caulker                              0.0      39.2
     Gladden Spit and Silk Cayes               0.0      99.9
     Glover's Reef                             0.0     320.7
     Hol Chan                                  0.0     414.1
     Port Honduras                             0.0     403.9
     Sapodilla Cayes                           0.0    1287.5
     South Water Caye                          0.0     509.5
     Turneffe Atolls                           0.0    1471.4
HND  Sistema de Humedales de Santa Elena      15.4      15.4

### Other points to raise

- **Biosphere reserves.** `UNESCO-MAB Biosphere Reserve` sites include buffer and transition zones where people live and work. Should they count as protected? (Protected Planet's coverage method, linked from the manual, is worth reading on this.)
- **Transboundary sites.** The manual says sites shared by countries carry several ISO3 codes, like `'CRI;PAN'`, so an exact match on `iso3 == 'CRI'` misses them. This subset has none, because it was built by keeping exact ISO3 matches, but the full WDPA does.
- **Points.** The full WDPA also has sites with no boundary, stored as points. This subset is polygons only.
- **The denominator.** The land areas come from Wikipedia and may handle lakes and inland water differently from the WDPA polygons.

In [20]:
mab = cam_pa['desig_eng'] == 'UNESCO-MAB Biosphere Reserve'
print(f"{mab.sum()} biosphere reserves, reported area {cam_pa.loc[mab, 'rep_area'].sum():,.0f} sq. km\n")

print("Records with several country codes:", wdpa['iso3'].str.contains(';').sum())

17 biosphere reserves, reported area 90,175 sq. km

Records with several country codes: 0


### How the analysis should change for terrestrial coverage

1. **Filter records**: drop `Proposed` sites (and decide on biosphere reserves).
2. **Keep land only**: drop predominantly marine sites (`marine == '2'`). Coastal sites (`'1'`) have a land part and a sea part, and separating them properly means clipping to a country land boundary, which this dataset does not include.
3. **Dissolve** per country so overlaps count once.
4. **Measure the polygons** in an equal-area projection instead of summing `rep_area`.
5. **Divide** by land area from a consistent source.

Without a land boundary, the cell below brackets the answer: a **lower bound** using only terrestrial sites, and an **upper bound** that adds coastal sites whole, sea part included.

In [21]:
def terrestrial_coverage(pa, marine_classes):
    """Dissolved protected area (sq. km) per country for the given marine classes,
    excluding Proposed sites, measured in an equal-area projection."""
    keep = (pa['status'] != 'Proposed') & pa['marine'].isin(marine_classes)
    flat = pa[keep].dissolve(by='iso3').to_crs(EQUAL_AREA)
    return flat.area / 1e6


cam['lower_sqkm'] = terrestrial_coverage(cam_pa, ['0']).reindex(cam.index, fill_value=0)
cam['upper_sqkm'] = terrestrial_coverage(cam_pa, ['0', '1']).reindex(cam.index, fill_value=0)
assert (cam['lower_sqkm'] <= cam['upper_sqkm'] + 1e-6).all()

cam['pct_lower'] = cam['lower_sqkm'] / cam['land_area_sqkm'] * 100
cam['pct_upper'] = cam['upper_sqkm'] / cam['land_area_sqkm'] * 100

cam[['country', 'pct_protected', 'pct_lower', 'pct_upper']].round(1)

,country,pct_protected,pct_lower,pct_upper
iso3,,,,
BLZ,Belize,44.3,35.4,35.8
CRI,Costa Rica,389.2,55.1,64.4
SLV,El Salvador,27.1,12.0,20.1
GTM,Guatemala,37.0,30.9,32.5
HND,Honduras,85.4,22.2,24.0
NIC,Nicaragua,65.2,34.1,42.5
PAN,Panama,177.1,27.2,36.7


> **Presenter note.** Costa Rica is still high. Ask students what is left in it: the biosphere reserves (for example La Amistad and Agua y Paz) are still counted, and coastal sites are counted whole in the upper bound.

**Take-away.** The colleague's percentages mixed land and sea, counted overlapping designations more than once, and included sites that are only proposed. Each fix needs a decision that depends on the question being asked, which is why the analysis should start from the manual rather than from `rep_area.sum()`.